# Part 3: NLP and Sequence Modeling Mini Project

This notebook loads a customer support text dataset, preprocesses the text, compares TF-IDF vectorization with sequence modeling, builds a baseline classifier, and explains a simple sequence architecture.

## Task 1: Dataset Understanding

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = 'customer_support_text_classification.csv'
df = pd.read_csv(DATA_PATH)

print('Number of records:', len(df))
print('Columns:', df.columns.tolist())
print('Target labels:', df['sentiment_label'].unique())
print('Class distribution:')
print(df['sentiment_label'].value_counts(normalize=False))

df[['ticket_id', 'channel', 'customer_message', 'sentiment_label']].head(10)

Number of records: 1500
Columns: ['ticket_id', 'channel', 'customer_message', 'sentiment_label', 'word_count', 'urgent_flag']
Target labels: ['neutral' 'positive' 'negative']
Class distribution:
sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64


,ticket_id,channel,customer_message,sentiment_label
0,TKT00001,chat,I need information about the payment process. ...,neutral
1,TKT00002,phone,I need information about the payment process.,neutral
2,TKT00003,email,The refund process was fast and convenient. I ...,positive
3,TKT00004,social,My refund is still pending and this experience...,negative
4,TKT00005,chat,Please tell me how to update my account details.,neutral
5,TKT00006,social,I need help finding the invoice for my last or...,neutral
6,TKT00007,chat,I am satisfied with the plan and would recomme...,positive
7,TKT00008,chat,I want to understand the warranty terms for th...,neutral
8,TKT00009,phone,I need help finding the invoice for my last or...,neutral
9,TKT00010,phone,My refund is still pending and this experience...,negative


The dataset contains customer message text and a sentiment label with three classes: `neutral`, `positive`, and `negative`.

## Task 2: Text Preprocessing

In [2]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text: str) -> list[str]:
    tokens = text.split()
    tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    return tokens

df['clean_text'] = df['customer_message'].apply(clean_text)
df['tokens'] = df['clean_text'].apply(tokenize)
df['text_length'] = df['clean_text'].apply(lambda x: len(x.split()))

print('Average text length (words):', df['text_length'].mean())
print('Sample cleaned text:')
print(df[['customer_message', 'clean_text', 'tokens']].head(6).to_string(index=False))

Average text length (words): 12.722666666666667
Sample cleaned text:
                                                                                            customer_message                                                                                                clean_text                                                                                tokens
I need information about the payment process. My ticket number is 78732. Please respond as soon as possible. i need information about the payment process my ticket number is 78732 please respond as soon as possible [need, information, payment, process, ticket, number, 78732, respond, soon, possible]
                                                               I need information about the payment process.                                                              i need information about the payment process                                                 [need, information, payment, process]
                            

The text is lowercased, non-alphanumeric symbols are removed, and simple tokenization is applied. Stopwords are removed to keep the model focused on meaningful terms.

## Task 3: Text Vectorization

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

print('TF-IDF feature matrix shape:', X_tfidf.shape)
print('Example TF-IDF features:', tfidf.get_feature_names_out()[:15])

TF-IDF feature matrix shape: (1500, 667)
Example TF-IDF features: ['10347' '10565' '10632' '10783' '10841' '10973' '11045' '11058' '11213'
 '11482' '11855' '12223' '12238' '12408' '12727']


Text must be converted into numerical vectors because machine learning and neural network models operate on numbers. Vectorization preserves word presence, frequency, or ordering so the model can learn patterns from the text.

## Task 4: Baseline Model

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

label_map = {label: idx for idx, label in enumerate(sorted(df['sentiment_label'].unique()))}
df['label'] = df['sentiment_label'].map(label_map)

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['label'], stratify=df['label'], test_size=0.2, random_state=42)

baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

print('Baseline accuracy:', accuracy_score(y_test, y_pred))
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=sorted(label_map.keys())))

Baseline accuracy: 1.0
Classification report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        99
     neutral       1.00      1.00      1.00       105
    positive       1.00      1.00      1.00        96

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



In [5]:
evaluation = pd.DataFrame([
    {
        'model': 'Logistic Regression + TF-IDF',
        'accuracy': accuracy_score(y_test, y_pred)
    }
])
evaluation.to_csv('results/model_evaluation.csv', index=False)

sample_df = X_test[:10].copy() if False else None
# Save a simple set of sample predictions to results/sample_predictions.txt
with open('results/sample_predictions.txt', 'w') as f:
    f.write('True label\tPredicted label\tText\n')
    for true_label, pred_label, text in zip(y_test[:10], y_pred[:10], df.loc[y_test.index[:10], 'customer_message']):
        f.write(f'{sorted(label_map.keys())[true_label]}\t{sorted(label_map.keys())[pred_label]}\t{text}\n')

The baseline model uses TF-IDF vectorization and logistic regression to classify sentiment. This gives a simple benchmark before sequence-based modeling.

## Task 5: Sequence Model Architecture

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

MAX_WORDS = 5000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'])
sequences = tokenizer.texts_to_sequences(df['clean_text'])
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print('Example sequence:', sequences[0])
print('Padded example:', padded_sequences[0])

Example sequence: [5, 30, 137, 40, 2, 91, 34, 4, 7, 8, 3, 185, 11, 13, 9, 14, 9, 15]
Padded example: [  5  30 137  40   2  91  34   4   7   8   3 185  11  13   9  14   9  15
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0]


In [7]:
X_seq_train, X_seq_test, y_seq_train, y_seq_test = train_test_split(padded_sequences, df['label'], stratify=df['label'], test_size=0.2, random_state=42)

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(len(label_map), activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_seq_train, y_seq_train, epochs=6, batch_size=32, validation_split=0.15, verbose=1)
seq_loss, seq_acc = model.evaluate(X_seq_test, y_seq_test, verbose=0)
print('Sequence model accuracy:', seq_acc)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-16 12:22:55.060849: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-05-16 12:22:55.060910: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-05-16 12:22:55.060916: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-05-16 12:22:55.061179: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-16 12:22:55.061192: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/6


2026-05-16 12:22:55.742112: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.3255 - loss: 1.1021 - val_accuracy: 0.3278 - val_loss: 1.1069
Epoch 2/6
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.3353 - loss: 1.0982 - val_accuracy: 0.3500 - val_loss: 1.0975
Epoch 3/6
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.3363 - loss: 1.1007 - val_accuracy: 0.3278 - val_loss: 1.1029
Epoch 4/6
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.4559 - loss: 1.0056 - val_accuracy: 0.6667 - val_loss: 0.5930
Epoch 5/6
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6225 - loss: 0.7647 - val_accuracy: 0.6056 - val_loss: 0.7387
Epoch 6/6
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6441 - loss: 0.5623 - val_accuracy: 0.6778 - val_loss: 0.4641
Sequence model accuracy: 0.6700000166893005


### Sequence Model Explanation
- Input sequence: tokenized text mapped to integer word IDs and padded/truncated to fixed length.
- Embedding layer: converts each word ID into a dense vector that captures semantic information.
- Recurrent layer: LSTM processes each token in order, maintaining memory across the sequence.
- Output layer: a softmax classifier over the sentiment classes.
- Loss function: `sparse_categorical_crossentropy` for multi-class classification.
- Evaluation metric: accuracy, supported by classification recall/precision as needed.

## Task 6: Attention and Transformer Reflection

- RNNs struggle with long-term dependencies because gradients can vanish or explode as information passes through many time steps, making it hard to learn relationships between distant tokens.
- LSTMs help with memory by using gated cells (input, forget, and output gates) that decide what information to keep, update, and output, which improves long-range dependency learning.
- Attention solves sequence-to-sequence tasks by allowing the model to focus on the most relevant parts of the source sequence for each output step, rather than compressing all information into a single fixed-size vector.
- Transformers are important because they use self-attention to process all tokens in parallel, capture global context more effectively, and scale to large models for modern NLP and generative AI.